In [1]:
# ============================================================
# NBA SPORTS GAMBLING PROJECT
# Collaborative Filtering using Matrix Factorization
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
model_data = pd.read_csv("../../data/processed/model_features.csv")

print("Dataset loaded!")
print(model_data.shape)

Dataset loaded!
(29575, 41)


In [3]:
required_columns = [
    "GAME_ID",
    "SEASON",
    "TEAM_ID_home",
    "TEAM_ID_away",
    "TOTAL_POINTS"
]

for column in required_columns:
    print(column, "->", column in model_data.columns)

GAME_ID -> True
SEASON -> True
TEAM_ID_home -> True
TEAM_ID_away -> True
TOTAL_POINTS -> True


In [4]:
train_data = model_data[
    model_data["SEASON"].isin([2012, 2013, 2014, 2015])
].copy()

validation_data = model_data[
    model_data["SEASON"] == 2016
].copy()

test_data = model_data[
    model_data["SEASON"] == 2017
].copy()

print("Training games:", len(train_data))
print("Validation games:", len(validation_data))
print("Testing games:", len(test_data))

Training games: 5681
Validation games: 1405
Testing games: 1382


In [5]:
all_teams = sorted(
    set(train_data["TEAM_ID_home"]) |
    set(train_data["TEAM_ID_away"])
)

team_to_index = {
    team_id: index
    for index, team_id in enumerate(all_teams)
}

num_teams = len(all_teams)

print("Number of teams:", num_teams)

Number of teams: 30


In [6]:
team_matrix_df = train_data.pivot_table(
    index="TEAM_ID_home",
    columns="TEAM_ID_away",
    values="TOTAL_POINTS",
    aggfunc="mean"
)

team_matrix_df = team_matrix_df.reindex(
    index=all_teams,
    columns=all_teams
)

print("Matrix shape:", team_matrix_df.shape)
print("Observed matchups:", team_matrix_df.notna().sum().sum())

Matrix shape: (30, 30)
Observed matchups: 870


In [7]:
training_mean = np.nanmean(team_matrix_df.values)

team_matrix_filled = team_matrix_df.fillna(training_mean).values

print("Missing values before:", team_matrix_df.isna().sum().sum())
print("Missing values after:", np.isnan(team_matrix_filled).sum())
print("Training mean:", round(training_mean, 2))

Missing values before: 30
Missing values after: 0
Training mean: 200.5


## Singular Value Decomposition (SVD)

Singular Value Decomposition (SVD) is used to decompose the team matchup matrix into latent factors.

In this project, SVD is applied to the Home Team × Away Team matrix created from historical NBA game total points.

The decomposition helps reconstruct the matrix using a smaller number of latent dimensions, which can be used to estimate the expected total points for team matchups.

In [8]:
U, S, Vt = np.linalg.svd(
    team_matrix_filled,
    full_matrices=False
)

print("U shape:", U.shape)
print("S shape:", S.shape)
print("Vt shape:", Vt.shape)

U shape: (30, 30)
S shape: (30,)
Vt shape: (30, 30)


## Latent Dimension Reduction

To reconstruct the matchup matrix using a lower-dimensional representation, we retain the first 10 singular values and corresponding singular vectors.


In [9]:
hidden_size = 10

U_k = U[:, :hidden_size]
S_k = S[:hidden_size]
Vt_k = Vt[:hidden_size, :]

print("U_k shape:", U_k.shape)
print("S_k shape:", S_k.shape)
print("Vt_k shape:", Vt_k.shape)


U_k shape: (30, 10)
S_k shape: (10,)
Vt_k shape: (10, 30)


## Reconstructing the Predicted Matchup Matrix

The reduced SVD components are multiplied together to reconstruct the team matchup matrix. Each cell represents the estimated total points for a home-team and away-team matchup.


In [10]:
predicted_matrix = U_k @ np.diag(S_k) @ Vt_k

print("Predicted matrix shape:", predicted_matrix.shape)

Predicted matrix shape: (30, 30)


## Collaborative Filtering Prediction

The reconstructed SVD matrix provides predicted total points for each home-team and away-team matchup.

For each validation game, the corresponding home-team and away-team indices are used to obtain the predicted total points from the reconstructed matrix.

In [11]:
cf_validation_predictions = []

for _, row in validation_data.iterrows():
    home_team = row["TEAM_ID_home"]
    away_team = row["TEAM_ID_away"]

    home_index = team_to_index[home_team]
    away_index = team_to_index[away_team]

    prediction = predicted_matrix[home_index, away_index]
    cf_validation_predictions.append(prediction)

cf_validation_predictions = np.array(cf_validation_predictions)

print("Number of validation predictions:", len(cf_validation_predictions))
print("First 5 predictions:", cf_validation_predictions[:5])

Number of validation predictions: 1405
First 5 predictions: [197.05864134 211.05798258 191.97132674 198.31235708 193.32631842]


## Collaborative Filtering Validation Performance

Mean Squared Error (MSE) is used to measure how close the Collaborative Filtering predictions are to the actual total points.

A lower MSE indicates better prediction performance.

In [12]:
cf_validation_mse = mean_squared_error(
    validation_data["TOTAL_POINTS"],
    cf_validation_predictions
)

print("Collaborative Filtering Validation MSE:", round(cf_validation_mse, 2))

Collaborative Filtering Validation MSE: 497.53


## Collaborative Filtering Test Prediction

After validating the Collaborative Filtering model, we now generate predictions for the unseen test season.

The test data is used only for final evaluation of the model.

In [13]:
cf_test_predictions = []

for _, row in test_data.iterrows():
    home_team = row["TEAM_ID_home"]
    away_team = row["TEAM_ID_away"]

    home_index = team_to_index[home_team]
    away_index = team_to_index[away_team]

    prediction = predicted_matrix[home_index, away_index]
    cf_test_predictions.append(prediction)

cf_test_predictions = np.array(cf_test_predictions)

print("Number of test predictions:", len(cf_test_predictions))
print("First 5 predictions:", cf_test_predictions[:5])

Number of test predictions: 1382
First 5 predictions: [211.23683619 208.40229424 207.21108846 200.36352801 198.31235708]


## Collaborative Filtering Test Performance

The Collaborative Filtering model is now evaluated on the unseen test data using Mean Squared Error (MSE).

This value will be used later to compare Collaborative Filtering with the other models.

In [14]:
cf_test_mse = mean_squared_error(
    test_data["TOTAL_POINTS"],
    cf_test_predictions
)

print("Collaborative Filtering Test MSE:", round(cf_test_mse, 2))

Collaborative Filtering Test MSE: 554.48


## Long Short-Term Memory (LSTM)

Long Short-Term Memory (LSTM) is a recurrent neural network that can learn patterns from sequential data.

In this project, the LSTM model is used to predict the total points scored in an NBA game using the available team features.

The LSTM model is trained using the training season data and evaluated on the validation and test seasons.

In [15]:
lstm_features = [
    "HOME_LAST3_POINTS_SCORED",
    "HOME_LAST3_POINTS_ALLOWED",
    "HOME_LAST3_FG_PCT",
    "HOME_LAST3_FT_PCT",
    "HOME_LAST3_FG3_PCT",
    "HOME_LAST3_AST",
    "HOME_LAST3_REB",
    "HOME_LAST3_POINT_DIFFERENTIAL",
    "HOME_DAYS_SINCE_LAST_GAME",
    "AWAY_LAST3_POINTS_SCORED",
    "AWAY_LAST3_POINTS_ALLOWED",
    "AWAY_LAST3_FG_PCT",
    "AWAY_LAST3_FT_PCT",
    "AWAY_LAST3_FG3_PCT",
    "AWAY_LAST3_AST",
    "AWAY_LAST3_REB",
    "AWAY_LAST3_POINT_DIFFERENTIAL",
    "AWAY_DAYS_SINCE_LAST_GAME"
]

X_train_lstm = train_data[lstm_features]
y_train_lstm = train_data["TOTAL_POINTS"]

X_validation_lstm = validation_data[lstm_features]
y_validation_lstm = validation_data["TOTAL_POINTS"]

X_test_lstm = test_data[lstm_features]
y_test_lstm = test_data["TOTAL_POINTS"]

print("Training features:", X_train_lstm.shape)
print("Validation features:", X_validation_lstm.shape)
print("Test features:", X_test_lstm.shape)

Training features: (5681, 18)
Validation features: (1405, 18)
Test features: (1382, 18)


In [16]:
from sklearn.preprocessing import StandardScaler

lstm_scaler = StandardScaler()

X_train_lstm_scaled = lstm_scaler.fit_transform(X_train_lstm)
X_validation_lstm_scaled = lstm_scaler.transform(X_validation_lstm)
X_test_lstm_scaled = lstm_scaler.transform(X_test_lstm)

print("Training scaled shape:", X_train_lstm_scaled.shape)
print("Validation scaled shape:", X_validation_lstm_scaled.shape)
print("Test scaled shape:", X_test_lstm_scaled.shape)

Training scaled shape: (5681, 18)
Validation scaled shape: (1405, 18)
Test scaled shape: (1382, 18)


In [17]:
X_train_lstm_seq = X_train_lstm_scaled.reshape(
    X_train_lstm_scaled.shape[0], 1, X_train_lstm_scaled.shape[1]
)

X_validation_lstm_seq = X_validation_lstm_scaled.reshape(
    X_validation_lstm_scaled.shape[0], 1, X_validation_lstm_scaled.shape[1]
)

X_test_lstm_seq = X_test_lstm_scaled.reshape(
    X_test_lstm_scaled.shape[0], 1, X_test_lstm_scaled.shape[1]
)

print("Training LSTM shape:", X_train_lstm_seq.shape)
print("Validation LSTM shape:", X_validation_lstm_seq.shape)
print("Test LSTM shape:", X_test_lstm_seq.shape)

Training LSTM shape: (5681, 1, 18)
Validation LSTM shape: (1405, 1, 18)
Test LSTM shape: (1382, 1, 18)


In [18]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
print("PyTorch ready!")

PyTorch version: 2.14.1+cpu
PyTorch ready!


In [19]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=18, hidden_size=20, dropout=0.2):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)

        x = hidden[-1]
        x = self.dropout(x)
        x = self.fc(x)

        return x


lstm_model = LSTMModel(
    input_size=18,
    hidden_size=20,
    dropout=0.2
)

print(lstm_model)

LSTMModel(
  (lstm): LSTM(18, 20, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=20, out_features=1, bias=True)
)


In [20]:
X_train_tensor = torch.tensor(
    X_train_lstm_seq,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_lstm.values,
    dtype=torch.float32
).reshape(-1, 1)

print("X train tensor shape:", X_train_tensor.shape)
print("y train tensor shape:", y_train_tensor.shape)

X train tensor shape: torch.Size([5681, 1, 18])
y train tensor shape: torch.Size([5681, 1])


In [21]:
criterion = nn.MSELoss()

optimizer = torch.optim.SGD(
    lstm_model.parameters(),
    lr=0.05
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: MSELoss()
Optimizer: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.05
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [22]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=10,
    shuffle=True
)

num_epochs = 100

for epoch in range(num_epochs):
    lstm_model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        predictions = lstm_model(X_batch)

        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        average_loss = epoch_loss / len(train_loader)
        print(
            f"Epoch [{epoch + 1}/{num_epochs}], "
            f"Loss: {average_loss:.4f}"
        )

print("LSTM training completed!")

Epoch [10/100], Loss: 356.9656
Epoch [20/100], Loss: 342.5923
Epoch [30/100], Loss: 344.7312
Epoch [40/100], Loss: 335.8986
Epoch [50/100], Loss: 334.6652
Epoch [60/100], Loss: 328.0398
Epoch [70/100], Loss: 332.1132
Epoch [80/100], Loss: 328.0612
Epoch [90/100], Loss: 322.3600
Epoch [100/100], Loss: 322.9867
LSTM training completed!


In [23]:
X_validation_tensor = torch.tensor(
    X_validation_lstm_seq,
    dtype=torch.float32
)

y_validation_tensor = torch.tensor(
    y_validation_lstm.values,
    dtype=torch.float32
).reshape(-1, 1)

print("X validation tensor shape:", X_validation_tensor.shape)
print("y validation tensor shape:", y_validation_tensor.shape)

X validation tensor shape: torch.Size([1405, 1, 18])
y validation tensor shape: torch.Size([1405, 1])


In [24]:
lstm_model.eval()

with torch.no_grad():
    validation_predictions = lstm_model(
        X_validation_tensor
    )

lstm_validation_predictions = validation_predictions.numpy().flatten()

print("Number of validation predictions:", len(lstm_validation_predictions))
print("First 5 predictions:", lstm_validation_predictions[:5])

Number of validation predictions: 1405
First 5 predictions: [193.6848  191.59595 195.10402 194.10841 189.98563]


In [25]:
lstm_validation_mse = mean_squared_error(
    y_validation_lstm,
    lstm_validation_predictions
)

print(
    "LSTM Validation MSE:",
    round(lstm_validation_mse, 2)
)

LSTM Validation MSE: 384.83


In [26]:
X_test_tensor = torch.tensor(
    X_test_lstm_seq,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test_lstm.values,
    dtype=torch.float32
).reshape(-1, 1)

print("X test tensor shape:", X_test_tensor.shape)
print("y test tensor shape:", y_test_tensor.shape)

X test tensor shape: torch.Size([1382, 1, 18])
y test tensor shape: torch.Size([1382, 1])


In [27]:
lstm_model.eval()

with torch.no_grad():
    test_predictions = lstm_model(
        X_test_tensor
    )

lstm_test_predictions = test_predictions.numpy().flatten()

print("Number of test predictions:", len(lstm_test_predictions))
print("First 5 predictions:", lstm_test_predictions[:5])

Number of test predictions: 1382
First 5 predictions: [206.43651 195.10402 194.1084  195.10233 195.37285]


In [28]:
lstm_test_mse = mean_squared_error(
    y_test_lstm,
    lstm_test_predictions
)

print(
    "LSTM Test MSE:",
    round(lstm_test_mse, 2)
)

LSTM Test MSE: 403.13
